---
title: Virtualize NISAR GUNW (HTTPS)
---

# Virtualize NISAR GUNW via HTTPS

This notebook virtualizes a NISAR granule over HTTPS instead of S3
direct access, so it works from anywhere (no us-west-2 requirement).

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning, module="earthaccess")

warnings.filterwarnings("ignore", message="Numcodecs codecs are not in the Zarr")
warnings.filterwarnings(
    "ignore", category=UserWarning, message=".*does not have a Zarr V3 specification.*"
)

## Find a NISAR GUNW granule

In [ ]:
import earthaccess

earthaccess.login()

results = earthaccess.search_data(
    short_name="NISAR_L2_GUNW_BETA_V1",
    point=(174.1192, -39.3379),  # lon, lat
    temporal=("2026-01-01", "2026-01-04"),
)
print(f"Found {len(results)} granule(s)")

granule = results[0]
https_url = [
    link for link in granule.data_links(access="external") if link.endswith(".h5")
][0]
print(f"HTTPS URL: {https_url}")

## Create virtual references with VirtualiZarr

In [ ]:
from urllib.parse import urlparse

import virtualizarr as vz
from obspec_utils.stores import AiohttpStore
from obspec_utils.registry import ObjectStoreRegistry

parsed = urlparse(https_url)
base_url = f"{parsed.scheme}://{parsed.netloc}"
token = earthaccess.get_edl_token()["access_token"]

store = AiohttpStore(
    base_url,
    headers={"Authorization": f"Bearer {token}"},
)
registry = ObjectStoreRegistry({base_url: store})

parser = vz.parsers.HDFParser()
vdt = vz.open_virtual_datatree(
    url=https_url,
    parser=parser,
    registry=registry,
)
vdt

## Writing to Icechunk (in progress)

Writing HTTPS-based virtual references to Icechunk requires Icechunk
to support earthaccess authentication when resolving virtual chunks.
This is under active development. For now, use the S3 version of this
notebook (`01-virtualize`) to write to Icechunk.

Track progress: https://github.com/earth-mover/icechunk/issues/997